In [ ]:
# C9-LHC Worker v0.1
# Dataset: Higgs → 4μ
import os, json, time
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')
C9_OUT = '/content/drive/MyDrive/c9_lhc_outputs'
os.makedirs(C9_OUT, exist_ok=True)
RUN_ID = '2026-07-20T21-23-17.116919+00-00'
print(f'C9-LHC Worker: {RUN_ID}')


In [ ]:
!pip install -q uproot awkward vector matplotlib numpy scipy
print('Ready')


In [ ]:
import urllib.request
URL = 'http://opendata.cern.ch/record/12341/files/Run2012A_DoubleMuParked.root'
NAME = 'higgs_4mu_2012A.root'
PATH = f'/content/{NAME}'
print(f'Downloading {NAME}...')
urllib.request.urlretrieve(URL, PATH)
print(f'Size: {os.path.getsize(PATH)/1e6:.1f} MB')


In [ ]:
import uproot, numpy as np, matplotlib.pyplot as plt
file = uproot.open(PATH)
print('Keys:', file.keys()[:5])
tree = None
for k in file.keys():
    if 'Tree' in str(type(file[k])) or 'TTree' in str(type(file[k])):
        tree = file[k]
        break
if tree:
    print(f'Tree: {tree.name}, Entries: {tree.num_entries}')
    print(f'Branches: {len(tree.keys())}')
else:
    print('No tree found')


In [ ]:
results = {
    'run_id': RUN_ID,
    'dataset': 'higgs_4mu_2012A',
    'channel': 'Higgs → 4μ',
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'status': 'completed',
    'events_processed': tree.num_entries if tree else 0,
    'c9_metadata': {'assembly_index_estimate': 0.0, 'priority_flag': False}
}
out = f'{C9_OUT}/c9_lhc_{dataset_key}_{RUN_ID}.json'
with open(out, 'w') as f: json.dump(results, f, indent=2)
print(f'Saved: {out}')
print(json.dumps(results, indent=2))


In [ ]:
done = {'run_id': RUN_ID, 'dataset': 'higgs_4mu_2012A', 'status': 'complete'}
with open(f'{C9_OUT}/COMPLETED_{RUN_ID}.json', 'w') as f:
    json.dump(done, f)
print('C9-LHC WORKER COMPLETE')
